<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/chipsff_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

Wait for a few seconds to restart the kernel

Install all necessary packages, including CHIPS-FF (https://github.com/usnistgov/chipsff)

In [ ]:
%%time
!conda install chgnet phonopy pytorch torchvision torchaudio pytorch-cuda -c pytorch -c nvidia -y --quiet

In [ ]:
!conda install ipykernel -q -y
!pip install  -q -U git+https://github.com/usnistgov/chipsff.git@develop
!pip install -q elastic
!pip install "plotly<6" "kaleido<1"

Run a sample geometry optimization using CHGNet and apply isotropic strain (energy vs. volume).

Let's test Silicon (JVASP-1002) and Gallium Arsenide (JVASP-1174)

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_optim.json
!run_chipsff --input_file input_optim.json

Let's take a look at the output files that were created (detailed json and log files with MLFF and ground truth DFT, automated error metrics)

In [ ]:
!ls JVASP-1174_chgnet/*

Energy vs. Volume Curve (Murnaghan Fit)

In [ ]:
from IPython.display import Image, display
display(Image("JVASP-1174_chgnet/E_vs_V_curve.png", width=800))

Let's create parity plots for lattice constant predictions (with respect to JARVIS-DFT results)

In [ ]:
from jarvis.db.jsonutils import loadjson
import glob
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_error

mem = []
ids = []
for filepath in glob.glob("*chgnet/*results.json"):
    d = loadjson(filepath)
    mem.append(d)
    ids.append(filepath.split("_")[0])

df = pd.DataFrame(mem)
df["id"] = ids
df.head()

initial_a = [entry["energy"]["initial_a"] for entry in mem]
initial_b = [entry["energy"]["initial_b"] for entry in mem]
initial_c = [entry["energy"]["initial_c"] for entry in mem]

final_a = [entry["energy"]["final_a"] for entry in mem]
final_b = [entry["energy"]["final_b"] for entry in mem]
final_c = [entry["energy"]["final_c"] for entry in mem]

fig_a = px.scatter(
    x=initial_a,
    y=final_a,
    labels={
        "x": "JARVIS-DFT a (Å)",
        "y": "MLFF a (Å)"
    },
    title="Lattice parameter a: JARVIS-DFT vs MLFF for CHGNet"
)
fig_a.show()

mae_a = mean_absolute_error(initial_a, final_a)
print(f"MAE (a): {mae_a:.3f} Å")


fig_b = px.scatter(
    x=initial_b,
    y=final_b,
    labels={
        "x": "JARVIS-DFT b (Å)",
        "y": "MLFF b (Å)"
    },
    title="Lattice parameter b: JARVIS-DFT vs MLFF for CHGNet"
)
fig_b.show()

mae_b = mean_absolute_error(initial_b, final_b)
print(f"MAE (b): {mae_b:.3f} Å")

fig_c = px.scatter(
    x=initial_c,
    y=final_c,
    labels={
        "x": "JARVIS-DFT c (Å)",
        "y": "MLFF c (Å)"
    },
    title="Lattice parameter c: JARVIS-DFT vs MLFF for CHGNet"
)
fig_c.show()

mae_c = mean_absolute_error(initial_c, final_c)
print(f"MAE (c): {mae_c:.3f} Å")

Let's run an example phonon calculation for Silicon using CHGNet

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_phonon.json
!run_chipsff --input_file input_phonon.json

From here, we can visualize the phonon band structure and thermal properties (free energy, heat capacity, entropy)

In [ ]:
from IPython.display import Image, display
display(Image("JVASP-1002_chgnet/ph_JVASP-1002_chgnet.png", width=800))
display(Image("JVASP-1002_chgnet/Thermal_Properties_JVASP-1002.png", width=800))

Let's calculate the vacancy formation energy with CHGNet for Si and GaAs

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_defect.json
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/chemical_potentials.json
!run_chipsff --input_file input_defect.json

In [ ]:
import glob, json
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_error

vac_records = []
for path in glob.glob("*chgnet/*results.json"):
    with open(path) as f:
        data = json.load(f)
    for rec in data.get("vacancy_energy", []):
        vac_records.append(
            {
                "name":   rec["name"],
                "JARVIS": rec["vac_en_entry"],
                "MLFF":   rec["vac_en"],
            }
        )

vac_df = pd.DataFrame(vac_records)

fig_vac = px.scatter(
    vac_df,
    x="JARVIS",
    y="MLFF",
    hover_name="name",
    labels={
        "JARVIS": "JARVIS-DFT vacancy formation energy (eV)",
        "MLFF":   "MLFF vacancy formation energy (eV)",
    },
    title="Vacancy formation energy: MLFF vs JARVIS-DFT for CHGNet",
)
lims = [vac_df[["JARVIS", "MLFF"]].min().min(),
        vac_df[["JARVIS", "MLFF"]].max().max()]
fig_vac.add_shape(type="line", x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1],
                  line=dict(dash="dash"))
fig_vac.show()

mae_vac = mean_absolute_error(vac_df["JARVIS"], vac_df["MLFF"])
print(f"MAE (vacancy energy): {mae_vac:.3f} eV")

Let's calculate the surface energy with CHGNet for Si for different crystal orientations [001], [111], [100]

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_surface.json
!run_chipsff --input_file input_surface.json

In [ ]:
import glob, json
import pandas as pd
import plotly.express as px
from sklearn.metrics import mean_absolute_error

surf_records = []
for path in glob.glob("*chgnet/*results.json"):
    with open(path) as f:
        data = json.load(f)
    for rec in data.get("surface_energy", []):
        surf_records.append(
            {
                "name":   rec["name"],
                "JARVIS": rec["surf_en_entry"],
                "MLFF":   rec["surf_en"],
            }
        )

surf_df = pd.DataFrame(surf_records)

fig_surf = px.scatter(
    surf_df,
    x="JARVIS",
    y="MLFF",
    hover_name="name",
    labels={
        "JARVIS": "JARVIS-DFT surface energy (J m⁻²)",
        "MLFF":   "MLFF surface energy (J m⁻²)",
    },
    title="Surface energy: MLFF vs JARVIS-DFT for CHGNet",
)
lims = [surf_df[["JARVIS", "MLFF"]].min().min(),
        surf_df[["JARVIS", "MLFF"]].max().max()]
fig_surf.add_shape(type="line", x0=lims[0], y0=lims[0], x1=lims[1], y1=lims[1],
                   line=dict(dash="dash"))
fig_surf.show()

mae_surf = mean_absolute_error(surf_df["JARVIS"], surf_df["MLFF"])
print(f"MAE (surface energy): {mae_surf:.3f} J m⁻²")

We can create amorphous Silicon by running a melt/quench Molecular Dynamics simulation with CHGNet

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_md_melt_quench.json
!run_chipsff --input_file input_md_melt_quench.json

Plot the radial distribution function (RDF) of amorphous Silicon

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from jarvis.core.atoms import Atoms
import glob

def get_rdf(atoms=None, ax=None, cutoff=10):
    dists = []
    for i in atoms.get_all_neighbors(r=cutoff):
        for j in i:
            dists.append(j[2])
    dists = np.array(dists)
    hist, bins = np.histogram(dists, bins=np.arange(0.1, cutoff + 0.1, 0.1))
    number_density = atoms.num_atoms / atoms.volume
    shell_vol = (
                4.0
                / 3.0
                * np.pi
                * (np.power(bins[1:], 3) - np.power(bins[:-1], 3))
            )
    rdf = (
                hist / shell_vol / number_density / atoms.num_atoms
            )
    # Plot on the given axis
    ax.plot(bins[:-1], rdf)
    ax.set_xlim([0, cutoff])
    ax.set_xlabel('Distance (Angstrom)')
    ax.set_ylabel('RDF')
    ax.set_title(atoms.composition.reduced_formula)

for i in glob.glob("JVASP-1002_chgnet/POSCAR_JVASP-1002_quenched_chgnet.vasp"):
    fig, ax = plt.subplots()
    atoms = Atoms.from_poscar(i)
    get_rdf(atoms=atoms, ax=ax)
    plt.show()

Example of testing the scaling (system size vs. wall time) of a Cu supercell for CHGNet

In [ ]:
!wget https://raw.githubusercontent.com/usnistgov/chipsff/refs/heads/develop/chipsff/example_inputs/input_scaling.json
!run_chipsff --input_file input_scaling.json

In [ ]:
display(Image("scaling_analysis_Cu/scaling_test.png", width=800))

Excercise: Edit paramters in the input_*.json files and re-run:

 !run_chipsff --input_file input_*.json

 for specific examples